# Connect with Google drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import zipfile

PROJECT = Path(
    "/content/drive/MyDrive/Surface-Scratch-Detection"
).resolve()

# Unzip

In [ ]:
zip_path   = PROJECT / "data" / "scratch_v2_patches.zip"
output_dir = PROJECT / "data"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(output_dir)

print("Unzipped:", output_dir)

# Check Struture dataset

In [ ]:
!find "{PROJECT}/dataset/scratch_v2_patches" -maxdepth 2 -type d

In [ ]:
cp -r /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_v2_patches /content/scratch_v2_patches

# Remove empty file

In [ ]:
from pathlib import Path

root = Path("/content/scratch_v2_patches")

for split in ["train", "valid", "test"]:
    img_dir = root / split / "images"
    mask_dir = root / split / "masks"

    bad_images = [p for p in img_dir.glob("*") if p.stat().st_size == 0]
    print(split, "bad images:", len(bad_images))

    for img_path in bad_images:
        stem = img_path.stem
        print("Delete image:", img_path)
        img_path.unlink(missing_ok=True)

        for mask_path in mask_dir.glob(stem + ".*"):
            print("Delete mask:", mask_path)
            mask_path.unlink(missing_ok=True)

# Train model

In [ ]:
from pathlib import Path
import os, sys

PROJECT = Path("/content/drive/MyDrive/Surface-Scratch-Detection").resolve()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

print("CWD:", Path.cwd())
print("src exists:", (PROJECT / "src").exists())

In [ ]:
!python -m src.unet.train \
    --data /content/scratch_v2_patches \
    --epochs 40 \
    --batch_size 8 \
    --img_size 512 \
    --base_channel 32 \
    --norm group \
    --lr 0.0001 \
    --num_workers 2 \
    --save_dir checkpoints/unet_v4

# Download model

In [ ]:
from google.colab import files

model_path = (
    "/content/drive/MyDrive/Surface-Scratch-Detection/"
    "checkpoints/unet_v4/best.pth"
)

files.download(model_path)